In [1]:
%%configure -f
{
    "executorMemory": "1g",
    "executorCores": 1,
    "driverMemory": "1g",
    "driverCores": 1,
    "numExecutors": 1
}

StatementMeta(sparkpool, 54, -1, Finished, Available, Finished, False)

See https://go.microsoft.com/fwlink/?linkid=2170827


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

StatementMeta(sparkpool, 54, 3, Finished, Available, Finished, False)

In [5]:
# Step 1: Load data
fact_transactions = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_transactions")

# Step 2: Convert to Pandas 
df = fact_transactions.toPandas()

print(f"Data shape: {df.shape}")
print(df.head())

StatementMeta(sparkpool, 54, 5, Finished, Available, Finished, False)

Data shape: (102041, 20)
  transaction_id  customer_id  merchant_id  CARD_ID  device_id  LOCATION_ID  \
0     2025-03-08        24239        97028        1     363229           10   
1     2025-03-08        24239        97028        1     363229           10   
2     2025-01-17        77250        27515        1     480487           10   
3     2025-01-17        77250        27515        1     480487           10   
4     2025-01-17        77250        27515        1     480487           10   

      date_id  transaction_amount transaction_type  \
0  2025-03-08                 6.0              POS   
1  2025-03-08                 6.0              POS   
2  2025-01-17                 9.0              ATM   
3  2025-01-17                 9.0              ATM   
4  2025-01-17                 9.0              ATM   

  is_international_transaction fraud_label unusual_time_transaction  \
0                          Yes      Normal                       No   
1                          Yes   

In [6]:
# Check data
print("Missing values:")
print(df.isnull().sum())



StatementMeta(sparkpool, 54, 6, Finished, Available, Finished, False)

Missing values:
transaction_id                  0
customer_id                     0
merchant_id                     0
CARD_ID                         0
device_id                       0
LOCATION_ID                     0
date_id                         0
transaction_amount              0
transaction_type                0
is_international_transaction    0
fraud_label                     0
unusual_time_transaction        0
failed_transaction_count        0
distance_from_home              0
previous_fraud_count            0
daily_transaction_count         0
weekly_transaction_count        0
max_transaction_last_24h        0
account_balance                 0
load_timestamp                  0
dtype: int64


In [7]:
print("\nData types:")
print(df.dtypes)



StatementMeta(sparkpool, 54, 7, Finished, Available, Finished, False)


Data types:
transaction_id                          object
customer_id                              int32
merchant_id                              int32
CARD_ID                                  int32
device_id                                int32
LOCATION_ID                              int32
date_id                                 object
transaction_amount                     float64
transaction_type                        object
is_international_transaction            object
fraud_label                             object
unusual_time_transaction                object
failed_transaction_count               float64
distance_from_home                     float64
previous_fraud_count                   float64
daily_transaction_count                float64
weekly_transaction_count               float64
max_transaction_last_24h               float64
account_balance                        float64
load_timestamp                  datetime64[us]
dtype: object


In [8]:
print("\nTarget variable distribution:")
print(df['fraud_label'].value_counts())



StatementMeta(sparkpool, 54, 8, Finished, Available, Finished, False)


Target variable distribution:
fraud_label
Normal    97193
Fraud      4848
Name: count, dtype: int64


In [39]:
# Select features (X) and target (y)
features = ['transaction_amount', 'failed_transaction_count', 'distance_from_home', 
            'previous_fraud_count', 'daily_transaction_count', 'weekly_transaction_count', 
            'max_transaction_last_24h', 'account_balance','merchant_id','device_id']



StatementMeta(sparkpool, 54, 39, Finished, Available, Finished, False)

In [40]:
X = df[features]
y = (df['fraud_label'] == 'Fraud').astype(int)  # Convert to 0/1

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFraud cases: {y.sum()}")
print(f"Normal cases: {len(y) - y.sum()}")



StatementMeta(sparkpool, 54, 40, Finished, Available, Finished, False)


Features shape: (102041, 10)
Target shape: (102041,)

Fraud cases: 4848
Normal cases: 97193


In [41]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\nTrain size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)

StatementMeta(sparkpool, 54, 41, Finished, Available, Finished, False)


Train size: (71428, 10)
Test size: (30613, 10)
(71428, 10)


In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,classification_report
import matplotlib.pyplot as plt

StatementMeta(sparkpool, 54, 42, Finished, Available, Finished, False)

In [43]:
lr_model = LogisticRegression(max_iter=1000, 
    class_weight='balanced',
    random_state=42)
lr_model.fit(X_train_scaled,y_train)

StatementMeta(sparkpool, 54, 43, Finished, Available, Finished, False)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [44]:
# Predictions

y_pred = lr_model.predict(X_test_scaled)
print(y_pred)

StatementMeta(sparkpool, 54, 44, Finished, Available, Finished, False)

[1 1 0 ... 0 0 0]


In [45]:
lr_accuracy = accuracy_score(y_test, y_pred)
lr_precision = precision_score(y_test, y_pred)
lr_recall = recall_score(y_test, y_pred)
lr_f1 = f1_score(y_test, y_pred)


StatementMeta(sparkpool, 54, 45, Finished, Available, Finished, False)

In [46]:
print(f"Accuracy:  {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")
print(f"F1-Score:  {lr_f1:.4f}")


StatementMeta(sparkpool, 54, 46, Finished, Available, Finished, False)

Accuracy:  0.5172
Precision: 0.0562
Recall:    0.5487
F1-Score:  0.1020


In [47]:
cm_lr = confusion_matrix(y_test, y_pred)
print(cm_lr)

StatementMeta(sparkpool, 54, 47, Finished, Available, Finished, False)

[[14995 14089]
 [  690   839]]


In [48]:
print(classification_report(y_test, y_pred))

StatementMeta(sparkpool, 54, 48, Finished, Available, Finished, False)

              precision    recall  f1-score   support

           0       0.96      0.52      0.67     29084
           1       0.06      0.55      0.10      1529

    accuracy                           0.52     30613
   macro avg       0.51      0.53      0.39     30613
weighted avg       0.91      0.52      0.64     30613



In [146]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, class_weight='balanced',
    random_state=42)
dt_model.fit(X_train_scaled, y_train)
y_pred_train = dt_model.predict(X_train_scaled)
y_pred_test = dt_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred_test))

StatementMeta(sparkpool, 54, 146, Finished, Available, Finished, False)

              precision    recall  f1-score   support

           0       0.96      0.52      0.67     29084
           1       0.06      0.60      0.11      1529

    accuracy                           0.52     30613
   macro avg       0.51      0.56      0.39     30613
weighted avg       0.92      0.52      0.65     30613



In [142]:
import joblib
import os
os.makedirs('/mnt/user-data/outputs', exist_ok=True)
print(joblib.dump(dt_model, '/mnt/user-data/outputs/dt_model.pkl'))
joblib.dump(scaler,'/mnt/user-data/outputs/scaler.pkl' )

StatementMeta(sparkpool, 54, 142, Finished, Available, Finished, False)

['/mnt/user-data/outputs/dt_model.pkl']


['/mnt/user-data/outputs/scaler.pkl']

In [143]:
dt_model = joblib.load('/mnt/user-data/outputs/dt_model.pkl')
scaler = joblib.load('/mnt/user-data/outputs/scaler.pkl')

StatementMeta(sparkpool, 54, 143, Finished, Available, Finished, False)

In [144]:
df = pd.DataFrame({
    'transaction_amount': [455],
    'failed_transaction_count': [25],
    'distance_from_home': [20000],
    'previous_fraud_count': [10000000],
    'daily_transaction_count': [5],
    'weekly_transaction_count': [95],
    'max_transaction_last_24h': [3000],
    'account_balance': [50],
    'merchant_id': [501],
    'device_id': [363]
})

values = scaler.transform(df)
pred = dt_model.predict(values)
#prob = dt_model.predict_proba(values)[0]
print(pred)



StatementMeta(sparkpool, 54, 144, Finished, Available, Finished, False)

[0]
